In [2]:
from datetime import datetime
import pandas as pd
from PyDI.entitymatching import (
    StandardBlocker,
    StringComparator,
    DateComparator,
    RuleBasedMatcher,
)

def movie_dataframes():
    df1 = pd.DataFrame([
        {"id": "movie1", "title": "Star Wars IV", "director": "George Lucas", "date": datetime(1977, 5, 25)},
        {"id": "movie2", "title": "Star Wars V", "director": "Irvin Kershner", "date": datetime(1980, 5, 21)},
    ])
    df2 = pd.DataFrame([
        {"id": "movie3", "title": "Star Wars IV", "director": "Irvin Kershner", "date": datetime(1977, 5, 25)},
        {"id": "movie4", "title": "Star Wars IV", "director": "George Lucas", "date": datetime(1977, 5, 25)},
    ])
    return df1, df2

/Users/luca/PycharmProjects/PyDI/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [4]:
df1, df2 = movie_dataframes()
matcher = RuleBasedMatcher()
comparator = StringComparator("title", "levenshtein")

blocker = StandardBlocker(df1, df2, on=["title"], id_column="id")

correspondences = matcher.match(df_left=df1, df_right=df2, candidates=blocker, comparators=[comparator], threshold=0.7, id_column='id')

correspondences

,id1,id2,score,notes
0,movie1,movie3,1.0,comparators=1
1,movie1,movie4,1.0,comparators=1


In [8]:
print(correspondences)

      id1     id2  score          notes
0  movie1  movie3    1.0  comparators=1
1  movie1  movie4    1.0  comparators=1


In [16]:
for _, row in correspondences.iterrows():
    print(row['id1'])

movie1
movie1


In [ ]:
from PyDI.entitymatching.blocking.noblocking import NoBlocker

matcher = RuleBasedMatcher()
comparators = [
    StringComparator("title", "levenshtein"),
    StringComparator("director", "levenshtein"),
    DateComparator("date", max_days_difference=365*2),
]
blocker = NoBlocker(df_left=df1, df_right=df2, id_column='id')

correspondences = matcher.match(
    df_left=df1,
    df_right=df2,
    candidates=blocker,
    comparators=comparators,
    weights=[2.0, 1.0, 1.0],
    threshold=0.75,
    id_column='id',
)
expected_ids = {("movie1", "movie3"), ("movie1", "movie4")}
actual_ids = set((row["id1"], row["id2"]) for _, row in correspondences.iterrows())

print(actual_ids)
print(expected_ids)
print(correspondences)

{('movie1', 'movie4'), ('movie1', 'movie3')}
{('movie1', 'movie4'), ('movie1', 'movie3')}
      id1     id2     score          notes
0  movie1  movie3  0.767857  comparators=3
1  movie1  movie4  1.000000  comparators=3


In [20]:
DateComparator("date", max_days_difference=365*2).compare(
    {"date": datetime(1977, 5, 25)},
    {"date": datetime(1980, 5, 21)},
)

0.0